# Librerias

In [1]:
# BLOQUE B2.1 — Aplicación de los veredictos de B2 y contraste con B1

import json
import sys
from pathlib import Path
import pandas as pd


def localizar_constantes(inicio=None):
    """Devuelve la carpeta que contiene constantes.py.

    Busca desde la ubicación del cuaderno hacia arriba. En cada nivel prueba
    la carpeta misma y su subcarpeta 'Bloque A', porque el módulo vive dentro
    de Bloque A para que todo se reproduzca desde ahí hacia abajo.
    """
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in [inicio, *inicio.parents]:
        for candidata in (base, base / "Bloque A"):
            if (candidata / "constantes.py").is_file():
                return candidata
    raise FileNotFoundError(
        "No se encontró constantes.py subiendo desde " + str(inicio)
    )


RUTA_CONSTANTES = localizar_constantes()
if str(RUTA_CONSTANTES) not in sys.path:
    sys.path.insert(0, str(RUTA_CONSTANTES))

import constantes

# Se imprime cuál se cargó: si hubiera una copia vieja del módulo en otra
# carpeta, el error aparece aquí y no diez celdas más abajo con números raros.
print("constantes.py cargado desde:", RUTA_CONSTANTES / "constantes.py")
print("RUTA_PROYECTO:            ", constantes.RUTA_PROYECTO)
print("Salidas de este cuaderno: ", constantes.ruta_entregable("B2.1"))

constantes.py cargado desde: F:\Tesis\Bloque A\constantes.py
RUTA_PROYECTO:             F:\Tesis
Salidas de este cuaderno:  F:\Tesis\Bloque B\Entregable B2.1


# Configuración y entradas

In [2]:
# Celda 2 — Configuración, entradas y procedencia
#
# ORDEN DEL BLOQUE B (lineal, sin ciclos ni reejecuciones):
#   B1    clasifica con OSM y OCM; deja 6 casos sin verificar
#   B2    mide la escala urbana y resuelve esos casos con manzana catastral
#   B2.1  aplica los veredictos y documenta qué cambió respecto de B1

RUTA_B1 = constantes.ruta_entregable("B1")
RUTA_B2 = constantes.ruta_entregable("B2")
RUTA_A4 = constantes.ruta_entregable("A4")
RUTA_OUT = Path(constantes.ruta_entregable("B2.1"))
RUTA_OUT.mkdir(parents=True, exist_ok=True)

ARCH_CLASIFICACION = constantes.exigir_archivo(
    constantes.ruta_entregable("B1", "B1_clasificacion_cuatro_categorias.csv")
)
ARCH_PARES = constantes.exigir_archivo(
    constantes.ruta_entregable("B1", "B1_pares_primarios.csv")
)
ARCH_VEREDICTOS = constantes.exigir_archivo(
    constantes.ruta_entregable("B2", "B2_veredictos.json")
)
ARCH_MANIFIESTO_B2 = constantes.exigir_archivo(
    constantes.ruta_entregable("B2", "B2_manifiesto_insumos.json")
)
ARCH_A4 = constantes.ruta_entregable("A4", "A4_resultado_final_OCM_OSM.csv")

# -----------------------------------------------------------------------------
# Procedencia
#
# B2 dejó registrado en su manifiesto el hash de cada archivo que consumió,
# incluidos los crudos congelados de A2 y A3. Acá se recalculan los hashes de
# los mismos archivos derivados y se exige que coincidan: si no coinciden, B1
# se regeneró después de que B2 corriera y los veredictos ya no describen esta
# clasificación. Las entradas 'crudo::' se arrastran tal cual, porque son el
# eslabón que B2 ya validó contra las fuentes congeladas.
# -----------------------------------------------------------------------------
with open(ARCH_MANIFIESTO_B2, encoding="utf-8") as fh:
    MANIFIESTO_B2 = json.load(fh)

INSUMOS_DERIVADOS = {
    "B1_clasificacion_cuatro_categorias.csv": ARCH_CLASIFICACION,
    "B1_pares_primarios.csv": ARCH_PARES,
}

print("PROCEDENCIA")
print("-" * 78)

discrepancias = []
for nombre, ruta in INSUMOS_DERIVADOS.items():
    calculado = constantes.sha256_archivo(ruta)
    registrado = MANIFIESTO_B2.get(nombre)
    if registrado is None:
        estado = "no estaba en el manifiesto de B2"
        discrepancias.append(nombre)
    elif calculado == registrado:
        estado = "coincide con B2"
    else:
        estado = "NO COINCIDE con B2"
        discrepancias.append(nombre)
    print(f"  {nombre}\n    {calculado[:16]}…  {estado}")

if discrepancias:
    raise RuntimeError(
        "Los insumos cambiaron respecto de los que consumió B2: "
        + ", ".join(discrepancias)
        + ". Los veredictos de B2 no describen esta clasificación; "
        "hay que volver a correr B2 antes de aplicarlos."
    )

# El archivo de veredictos es la salida de B2, no un insumo suyo, así que no
# figura en su manifiesto. B2.1 registra su hash por cuenta propia para que
# quede amarrado a este resultado.
HASH_VEREDICTOS = constantes.sha256_archivo(ARCH_VEREDICTOS)
print(f"  B2_veredictos.json\n    {HASH_VEREDICTOS[:16]}…  registrado por B2.1")

CRUDOS_HEREDADOS = {
    k: v for k, v in MANIFIESTO_B2.items() if k.startswith("crudo::")
}
print(f"\n  Crudos congelados heredados de B2: {len(CRUDOS_HEREDADOS)}")
for k in CRUDOS_HEREDADOS:
    print(f"    {k.removeprefix('crudo::')}")

# -----------------------------------------------------------------------------
# Entradas
#
# Los identificadores se leen como texto a propósito. Con nulos presentes
# pandas infiere float y escribe '475409.0', que no une contra '475409'.
# -----------------------------------------------------------------------------
PREVIA = pd.read_csv(
    ARCH_CLASIFICACION,
    encoding="utf-8-sig",
    dtype={"ocm_id": "string", "osm_uid": "string", "caso": "string"},
)
PREVIA["ocm_id"] = PREVIA["ocm_id"].str.replace(r"\.0$", "", regex=True)

PARES = pd.read_csv(
    ARCH_PARES,
    encoding="utf-8-sig",
    dtype={"ocm_id": "string", "osm_uid": "string", "caso": "string"},
)
PARES["ocm_id"] = PARES["ocm_id"].str.replace(r"\.0$", "", regex=True)

VEREDICTOS, FICHA = constantes.cargar_veredictos_b2()

print("\n\nENTRADAS")
print("-" * 78)
print(f"  B1 · clasificación:  {len(PREVIA)} registros")
print(f"  B1 · pares primarios: {len(PARES)} pares")
print(f"  B2 · verificación:    {FICHA['resueltos']} de {FICHA['total']} casos resueltos")
print(f"       método:     {FICHA['metodo']}")
print(f"       fuente:     {FICHA['fuente']['fuente']} v{FICHA['fuente']['version']}")
print(f"       licencia:   {FICHA['fuente']['licencia']}")
print(f"       descarga:   {FICHA['fuente']['fecha_descarga']}")
print(f"       tolerancia: {FICHA['tolerancia_adoptada_m']} m "
      f"(banda evaluada: {FICHA['banda_tolerancia_m']} m)")
print(f"       área máxima de manzana admitida: "
      f"{FICHA['area_max_manzana_m2']:,.0f} m²")

# -----------------------------------------------------------------------------
# Consistencia entre las dos entradas
#
# Los casos declarados en B1 y los veredictos emitidos por B2 tienen que ser
# exactamente el mismo conjunto. Un caso en B1 sin veredicto quedaría sin
# resolver en silencio; un veredicto sin caso en B1 no se aplicaría a nada.
# -----------------------------------------------------------------------------
CASOS_B1 = set(PREVIA.loc[PREVIA["caso"].notna(), "caso"])
CASOS_B2 = set(VEREDICTOS)

assert CASOS_B1 == CASOS_B2, (
    f"Los casos no coinciden. Solo en B1: {sorted(CASOS_B1 - CASOS_B2)}; "
    f"solo en B2: {sorted(CASOS_B2 - CASOS_B1)}"
)
assert len(PREVIA) == 72, f"Se esperaban 72 registros de B1, hay {len(PREVIA)}"
assert FICHA["total"] == len(CASOS_B2), "El conteo de casos de B2 no cuadra con sus veredictos"
assert PREVIA["caso"].dropna().is_unique, "Hay un caso declarado en más de un registro de B1"

print(f"\n  Casos declarados por B1 y resueltos por B2: {len(CASOS_B1)} — coinciden")
print(f"  Con conclusión adoptada: "
      f"{sorted(c for c, v in VEREDICTOS.items() if v is not None)}")
print(f"  Sin conclusión (None):   "
      f"{sorted(c for c, v in VEREDICTOS.items() if v is None)}")

CASOS_EN_PARES = PARES[PARES["caso"].notna()].set_index("caso")
verificados = 0

for caso, fila in (PREVIA[PREVIA["caso"].notna()]
                   .set_index("caso")[["ocm_id", "osm_uid", "distancia_m"]]
                   .iterrows()):
    if caso not in CASOS_EN_PARES.index:
        continue
    par = CASOS_EN_PARES.loc[caso]
    assert par["ocm_id"] == fila["ocm_id"] and par["osm_uid"] == fila["osm_uid"], (
        f"{caso}: los dos archivos de B1 describen pares distintos"
    )
    assert abs(par["distancia_m"] - fila["distancia_m"]) < 0.1, (
        f"{caso}: la distancia difiere entre los dos archivos de B1 "
        f"({par['distancia_m']} vs {fila['distancia_m']})"
    )
    verificados += 1

print(f"  Coherencia entre los dos archivos de B1: verificada en "
      f"{verificados} casos (los tres N1 no son pares primarios)")

PROCEDENCIA
------------------------------------------------------------------------------
  B1_clasificacion_cuatro_categorias.csv
    4f6d8549d9d829f4…  coincide con B2
  B1_pares_primarios.csv
    5709dadedd1c6f65…  coincide con B2
  B2_veredictos.json
    b9e0d7ee9f101cb3…  registrado por B2.1

  Crudos congelados heredados de B2: 3
    A2_osm_crudo_Bogota_2026-08-29
    A2_osm_crudo_Medellin_2026-08-29
    A3_ocm_crudo_Colombia_estado-al-2026-08-29_bajado-2026-09-02


ENTRADAS
------------------------------------------------------------------------------
  B1 · clasificación:  72 registros
  B1 · pares primarios: 20 pares
  B2 · verificación:    2 de 6 casos resueltos
       método:     contención en manzana catastral UAECD (Mapa de Referencia IDECA), con tolerancia evaluada sobre una banda medida en la misma capa y criterio de suficiencia por área del polígono; solo se adoptan veredictos estables en toda la banda
       fuente:     Manzana. Bogotá D.C. (UAECD / Mapa de Referencia

# Reglas de reasignación

In [3]:
# =============================================================================
# Celda 3 — Reglas de reasignación
#
# Traduce cada veredicto de B2 en un cambio de categoría. Nada más: los
# registros sin caso declarado no se tocan.
#
# DOS DECISIONES DE DISEÑO, EXPLÍCITAS:
#
# 1. El estado de verificación sale del nombre de la categoría y pasa a la
#    columna 'verificacion_externa'. La categoría responde qué es la
#    estación; la columna, con cuánto respaldo se afirma. Mezclarlas en una
#    sola cadena obliga a cualquier bloque posterior a parsear texto para
#    agrupar, y parte una misma categoría en dos al contar.
#
# 2. Un veredicto negativo disuelve el par y ahí termina. El registro OSM
#    liberado NO se vuelve a emparejar contra otra OCM. El emparejamiento de
#    B1 es mutuo —cada registro eligió al otro como su más cercano— y
#    reabrirlo desde este cuaderno sería rehacer B1 por fuera de B1, que es
#    justo lo que la arquitectura lineal del proyecto prohíbe. Si alguna vez
#    se quiere reevaluar, se corrige B1 y se corre la cadena completa.
#
# NINGÚN VEREDICTO None SE TRATA COMO False. Que la manzana no discrimine no
# es evidencia de que sean estaciones distintas; es ausencia de evidencia.
# Convertirlo en un descarte sería afirmar algo que B2 explícitamente no
# concluyó.
# =============================================================================

SUFIJO_SIN_VERIFICAR = " (sin verificar)"

ESTADO_CONFIRMADO = "confirmado"
ESTADO_DESCARTADO = "descartado"
ESTADO_NO_CONCLUYENTE = "no concluyente"


def categoria_base(categoria):
    """Quita el sufijo de estado que B1 pegaba al nombre de la categoría."""
    if categoria.endswith(SUFIJO_SIN_VERIFICAR):
        return categoria[: -len(SUFIJO_SIN_VERIFICAR)]
    return categoria


def reasignar(categoria_previa, veredicto):
    """Devuelve (categoria, origen, estado) para un registro con caso.

    C3 — par lejano con evidencia textual, pendiente de decidir si las dos
         fuentes describen la misma estación:
         True  -> son la misma; el par se sostiene y queda como error de
                  posición entre fuentes.
         False -> son estaciones distintas; el par se disuelve y la OCM
                  queda sin contraparte. El registro OSM liberado lo
                  reincorpora la celda siguiente como fila propia.

    C5/N1 — punto OSM adicional cerca de una OCM ya emparejada:
         True  -> es otro punto de carga del mismo sitio.
         False -> es una instalación distinta que OCM no registra.

    'origen' es None cuando la categoría no cambia, para no reescribirlo.
    """
    base = categoria_base(categoria_previa)

    if veredicto is None:
        return base, None, ESTADO_NO_CONCLUYENTE

    if base.startswith("C3"):
        if veredicto:
            return "C3 Desacuerdo posicional", "AMBAS", ESTADO_CONFIRMADO
        return "C4 Sin contraparte (solo OCM)", "OCM", ESTADO_DESCARTADO

    if base.startswith("C5"):
        if veredicto:
            return "C5 Punto secundario del mismo sitio", "OSM->OCM", ESTADO_CONFIRMADO
        return "C4 Sin contraparte (solo OSM)", "OSM", ESTADO_DESCARTADO

    raise ValueError(
        f"Caso declarado sobre una categoría que no admite verificación: "
        f"{categoria_previa!r}. Solo C3 y C5 tienen casos."
    )


# -----------------------------------------------------------------------------
# Aplicación
# -----------------------------------------------------------------------------
ACT = PREVIA.copy()
ACT["categoria_previa"] = ACT["categoria"]
ACT["categoria"] = ACT["categoria"].map(categoria_base)
ACT["veredicto"] = pd.Series(pd.NA, index=ACT.index, dtype="object")
ACT["verificacion_externa"] = pd.Series(pd.NA, index=ACT.index, dtype="string")
ACT["motivo_verificacion"] = pd.Series(pd.NA, index=ACT.index, dtype="string")

for i, fila in ACT[ACT["caso"].notna()].iterrows():
    veredicto = VEREDICTOS[fila["caso"]]
    categoria, origen, estado = reasignar(fila["categoria_previa"], veredicto)

    ACT.at[i, "categoria"] = categoria
    ACT.at[i, "veredicto"] = veredicto
    ACT.at[i, "verificacion_externa"] = estado
    ACT.at[i, "motivo_verificacion"] = FICHA["motivos"][fila["caso"]]
    if origen is not None:
        ACT.at[i, "origen"] = origen

# -----------------------------------------------------------------------------
# Lectura de lo que acaba de pasar
# -----------------------------------------------------------------------------
print("REASIGNACIÓN CASO POR CASO")
print("-" * 78)
for _, r in ACT[ACT["caso"].notna()].sort_values("caso").iterrows():
    movio = "se mantiene" if r["categoria"] == categoria_base(r["categoria_previa"]) else "PASA A"
    print(f"\n  {r['caso']}  ·  {r['nombre_ocm']}")
    print(f"      veredicto B2: {r['veredicto']}  ({r['verificacion_externa']})")
    print(f"      de:     {categoria_base(r['categoria_previa'])}")
    print(f"      {movio}: {r['categoria']}")
    print(f"      razón:  {r['motivo_verificacion']}")

cambiaron = (ACT["categoria"] != ACT["categoria_previa"].map(categoria_base)).sum()
sufijo_restante = ACT["categoria"].str.endswith(SUFIJO_SIN_VERIFICAR).sum()

print("\n" + "-" * 78)
print(f"  Registros que cambian de categoría: {int(cambiaron)} de {len(ACT)}")
print(f"  Categorías que aún cargan el sufijo en el nombre: {int(sufijo_restante)}")
print(f"  Estado de verificación:")
print(ACT["verificacion_externa"].value_counts(dropna=False).to_string())

REASIGNACIÓN CASO POR CASO
------------------------------------------------------------------------------

  C3-1  ·  EnelX Nicolas de Federman
      veredicto B2: None  (no concluyente)
      de:     C3 Desacuerdo posicional
      se mantiene: C3 Desacuerdo posicional
      razón:  misma manzana 005106010 (20,404 m²) — NO ADOPTADO: el veredicto cambia dentro de la banda de tolerancia

  C3-2  ·  Parqueadero universidad EAN
      veredicto B2: False  (descartado)
      de:     C3 Desacuerdo posicional
      PASA A: C4 Sin contraparte (solo OCM)
      razón:  manzanas distintas 008311007 (20,586 m²) vs 008312017 (7,059 m²)

  C3-3  ·  EnelX Terminal Salitre
      veredicto B2: False  (descartado)
      de:     C3 Desacuerdo posicional
      PASA A: C4 Sin contraparte (solo OCM)
      razón:  manzanas distintas 006319011 (16,274 m²) vs 006319001 (154,881 m²)

  N1-1  ·  Compensar
      veredicto B2: None  (no concluyente)
      de:     C5 Punto secundario del mismo sitio
      se mantien

# Registros liberados al disolverse un par

In [4]:
# =============================================================================
# Celda 4 — Registros liberados al disolverse un par
#
# Cuando B2 descarta un caso C3, el par deja de existir. La fila que tenía
# B1 representaba a los dos registros juntos, así que ahora hay que separarla:
# la OCM se queda en su fila, ya reclasificada como «solo OCM» en la celda
# anterior, y el registro OSM tiene que aparecer por derecho propio como
# «solo OSM». Si no se hiciera, ese punto de carga desaparecería del
# inventario: dejaría de estar en el par sin entrar en ninguna otra parte.
#
# En C5 no aplica: ahí el registro OSM ya ocupaba su propia fila desde B1.
#
# Las columnas del par se vacían en la fila de la OCM. Conservar la distancia
# de un par que se acaba de disolver es guardar la medida de una relación que
# ya declaramos inexistente, y cualquiera que lea el archivo después la
# interpretaría como si todavía hubiera pareja.
# =============================================================================

COLUMNAS_DEL_PAR = [
    "osm_uid", "nombre_osm", "distancia_m",
    "similitud_nombre", "similitud_operador",
]

ACT["liberado_de_par"] = False

liberados = []
for i, fila in ACT[ACT["caso"].notna()].iterrows():
    if fila["verificacion_externa"] != ESTADO_DESCARTADO:
        continue
    if not categoria_base(fila["categoria_previa"]).startswith("C3"):
        continue

    nueva = {columna: pd.NA for columna in ACT.columns}
    nueva.update({
        "ciudad": fila["ciudad"],
        "categoria": "C4 Sin contraparte (solo OSM)",
        "origen": "OSM",
        "osm_uid": fila["osm_uid"],
        "nombre_osm": fila["nombre_osm"],
        "caso": fila["caso"],
        "categoria_previa": fila["categoria_previa"],
        "veredicto": fila["veredicto"],
        "verificacion_externa": ESTADO_DESCARTADO,
        "motivo_verificacion": (
            f"registro OSM liberado al disolverse el par {fila['caso']}"
        ),
        "liberado_de_par": True,
    })
    liberados.append(nueva)

    for columna in COLUMNAS_DEL_PAR:
        ACT.at[i, columna] = pd.NA

if liberados:
    NUEVOS = pd.DataFrame(liberados, columns=ACT.columns)
    # Los tipos se fijan a mano: si se dejan inferir, 'ocm_id' vuelve a
    # float y escribe '475409.0', que no une contra '475409'.
    for columna, tipo in ACT.dtypes.items():
        if pd.api.types.is_numeric_dtype(tipo) and tipo != bool:
            NUEVOS[columna] = pd.to_numeric(NUEVOS[columna], errors="coerce")
        elif tipo == "string":
            NUEVOS[columna] = NUEVOS[columna].astype("string")
    NUEVOS["liberado_de_par"] = NUEVOS["liberado_de_par"].astype(bool)
    ACT = pd.concat([ACT, NUEVOS], ignore_index=True)

ACT = (ACT.sort_values(["ciudad", "categoria", "distancia_m"], na_position="last")
          .reset_index(drop=True))

# -----------------------------------------------------------------------------
# Conteo de estaciones físicas
#
# Un registro no es una estación. Las filas C5 son puntos de carga adicionales
# de un sitio que ya está contado en su par, así que se descuentan. Ese es el
# mismo criterio con que B1 reportó 69 estaciones sobre 72 registros, y acá se
# aplica igual para que las dos cifras sean comparables.
# -----------------------------------------------------------------------------
def contar_estaciones_fisicas(tabla):
    secundarios = tabla["categoria"].str.startswith("C5").sum()
    return len(tabla) - int(secundarios)


fisicas_antes = contar_estaciones_fisicas(PREVIA)
fisicas_despues = contar_estaciones_fisicas(ACT)

print("REGISTROS LIBERADOS")
print("-" * 78)
for fila in liberados:
    print(f"  {fila['caso']}  {fila['osm_uid']}  ·  {fila['nombre_osm']}")
print(f"\n  Registros liberados: {len(liberados)}")
print(f"  Registros totales:   {len(PREVIA)} -> {len(ACT)}")

print("\nESTACIONES FÍSICAS")
print("-" * 78)
print(f"  Antes de la verificación (B1):    {fisicas_antes}")
print(f"  Después de la verificación (B2.1): {fisicas_despues}")

por_ciudad = (ACT.groupby("ciudad")
                 .apply(contar_estaciones_fisicas, include_groups=False)
                 .rename("estaciones"))
print("\n  Por ciudad:")
print(por_ciudad.to_string())

# Verificación independiente del conteo: se llega a la misma cifra por otro
# camino, contando registros únicos de cada fuente en vez de restar C5.
solo_ocm = ACT["ocm_id"].notna() & ACT["osm_uid"].isna()
solo_osm = ACT["ocm_id"].isna() & ACT["osm_uid"].notna()
emparejados = ACT["ocm_id"].notna() & ACT["osm_uid"].notna() & ~ACT["categoria"].str.startswith("C5")
control = int(solo_ocm.sum() + solo_osm.sum() + emparejados.sum())

assert control == fisicas_despues, (
    f"Los dos caminos de conteo no coinciden: {control} vs {fisicas_despues}"
)
print(f"\n  Verificación independiente del conteo: {control} — coincide")

REGISTROS LIBERADOS
------------------------------------------------------------------------------
  C3-2  node/13010000994  ·  Universidad EAN (Bogotá)
  C3-3  node/13009702030  ·  Punto de carga Terminal de Transportes Salitre

  Registros liberados: 2
  Registros totales:   72 -> 74

ESTACIONES FÍSICAS
------------------------------------------------------------------------------
  Antes de la verificación (B1):    69
  Después de la verificación (B2.1): 71

  Por ciudad:
ciudad
Bogota      52
Medellin    19

  Verificación independiente del conteo: 71 — coincide


# Contraste

In [5]:
print("=" * 78)
print("CONTRASTE  ·  clasificación de B1  frente a  clasificación verificada")
print("=" * 78)

# Las categorías de B1 se comparan por su nombre base, sin el sufijo de
# estado: si no, una misma categoría aparecería como dos filas distintas y
# el cambio se leería como si algo hubiera aparecido de la nada.
antes = PREVIA["categoria"].map(categoria_base).value_counts()
despues = ACT["categoria"].value_counts()

CONTRASTE = (pd.DataFrame({"B1 (sin verificar)": antes,
                           "B2.1 (verificado)": despues})
             .fillna(0).astype(int)
             .sort_index())
CONTRASTE["cambio"] = (CONTRASTE["B2.1 (verificado)"]
                       - CONTRASTE["B1 (sin verificar)"])

print(CONTRASTE.to_string())
print(f"\n  Totales: {CONTRASTE['B1 (sin verificar)'].sum()} registros "
      f"-> {CONTRASTE['B2.1 (verificado)'].sum()} registros")
print(f"  Estaciones físicas: {fisicas_antes} -> {fisicas_despues}")

# -----------------------------------------------------------------------------
# Movimientos caso por caso
#
# Solo las filas con caso declarado. Se excluyen las liberadas para que cada
# caso aparezca una vez: la fila liberada es consecuencia del movimiento, no
# un movimiento aparte.
# -----------------------------------------------------------------------------
MOVIMIENTOS = (ACT[ACT["caso"].notna() & ~ACT["liberado_de_par"]]
               [["caso", "ciudad", "nombre_ocm", "nombre_osm",
                 "categoria_previa", "categoria", "veredicto",
                 "verificacion_externa", "motivo_verificacion"]]
               .sort_values("caso")
               .reset_index(drop=True))
MOVIMIENTOS["categoria_previa"] = MOVIMIENTOS["categoria_previa"].map(categoria_base)
MOVIMIENTOS["cambio_de_categoria"] = (
    MOVIMIENTOS["categoria"] != MOVIMIENTOS["categoria_previa"]
)

resueltos = int((MOVIMIENTOS["verificacion_externa"] != ESTADO_NO_CONCLUYENTE).sum())
movidos = int(MOVIMIENTOS["cambio_de_categoria"].sum())

print("\n" + "-" * 78)
print("MOVIMIENTOS POR CASO")
print("-" * 78)
print(f"  Casos declarados por B1:            {len(MOVIMIENTOS)}")
print(f"  Con conclusión adoptada por B2:     {resueltos}")
print(f"  Que cambian de categoría:           {movidos}")
print(f"  Que siguen sin conclusión:          {len(MOVIMIENTOS) - resueltos}")

# -----------------------------------------------------------------------------
# Conteos absolutos actualizados, por ciudad
# -----------------------------------------------------------------------------
CONTEOS = (ACT.groupby(["categoria", "ciudad"]).size()
           .unstack(fill_value=0))
CONTEOS["TOTAL"] = CONTEOS.sum(axis=1)

print("\n" + "-" * 78)
print("CONTEOS ABSOLUTOS DESPUÉS DE LA VERIFICACIÓN")
print("-" * 78)
print(CONTEOS.to_string())

# -----------------------------------------------------------------------------
# Cuánto del inventario sigue apoyado en una sola fuente
#
# Es el hallazgo central de la fase 1 y la verificación externa lo mueve, así
# que conviene recalcularlo acá y no arrastrar la cifra de B1.
# -----------------------------------------------------------------------------
una_sola_fuente = int(ACT["categoria"].str.startswith("C4").sum())
print(f"\n  Registros presentes en una sola fuente: "
      f"{una_sola_fuente} de {len(ACT)} "
      f"({una_sola_fuente / len(ACT):.1%})")
print(f"  En B1 eran: "
      f"{int(PREVIA['categoria'].str.startswith('C4').sum())} de {len(PREVIA)}")

# -----------------------------------------------------------------------------
# Verificación independiente del contraste
#
# La tabla de arriba se construyó contando categorías. Acá se reconstruye el
# mismo total por un camino distinto: partiendo de los 72 registros de B1 y
# sumando únicamente los liberados. Si los dos caminos no coinciden, algún
# registro se duplicó o se perdió en el proceso.
# -----------------------------------------------------------------------------
esperado = len(PREVIA) + len(liberados)
assert len(ACT) == esperado, (
    f"El total no cuadra: {len(ACT)} registros frente a "
    f"{len(PREVIA)} + {len(liberados)} liberados"
)
assert CONTRASTE["cambio"].sum() == len(liberados), (
    "La suma de los cambios por categoría no equivale a los registros nuevos"
)
print(f"\n  Verificación independiente del total: "
      f"{len(PREVIA)} + {len(liberados)} = {len(ACT)} — coincide")

CONTRASTE  ·  clasificación de B1  frente a  clasificación verificada
                                     B1 (sin verificar)  B2.1 (verificado)  cambio
categoria                                                                         
C1 Coincidencia confirmada                            5                  5       0
C2 Coincidencia probable                              9                  9       0
C3 Desacuerdo posicional                              3                  1      -2
C4 Sin contraparte (solo OCM)                        20                 22       2
C4 Sin contraparte (solo OSM)                        32                 34       2
C5 Punto secundario del mismo sitio                   3                  3       0

  Totales: 72 registros -> 74 registros
  Estaciones físicas: 69 -> 71

------------------------------------------------------------------------------
MOVIMIENTOS POR CASO
------------------------------------------------------------------------------
  Casos declar

# Descomposición del rojo de A4

In [6]:
# A4 está congelado y no se toca: es el contraste unidireccional, una fila por
# estación de OpenChargeMap, y se conserva como registro del método anterior.
# Esta celda NO lo recalcula. Hace un cruce por ocm_id entre los 38 registros
# de A4 y la categoría que esos mismos registros tienen hoy, después de la
# verificación externa. Es un 'map', no un método alternativo.
#
# Para qué sirve: A4 clasificó 24 registros como ROJO sin poder distinguir
# qué eran. La revisión de la tutoría señaló exactamente eso —que «las
# ubicaciones son muy diferentes» y «no hay coincidencia» son fenómenos
# distintos que llevan a conclusiones opuestas—. Esta tabla es la respuesta a
# esa observación, ya verificada contra una tercera fuente.
# =============================================================================

DESCOMPOSICION = None
if not Path(ARCH_A4).exists():
    print("No se encontró A4_resultado_final_OCM_OSM.csv; se omite la "
          "descomposición. El resto del cuaderno no depende de ella.")
else:
    A4 = pd.read_csv(
        ARCH_A4, encoding="utf-8-sig",
        dtype={"ocm_id": "string", "osm_uid": "string"},
    )
    A4["ocm_id"] = A4["ocm_id"].str.replace(r"\.0$", "", regex=True)

    # La tabla de referencia excluye las filas C5: son puntos secundarios que
    # comparten ocm_id con su par primario, y dejarlas haría que un mismo
    # ocm_id apuntara a dos categorías. El assert lo comprueba en vez de
    # confiar en que así sea.
    BASE = ACT[ACT["ocm_id"].notna() & ~ACT["categoria"].str.startswith("C5")]
    assert not BASE["ocm_id"].duplicated().any(), (
        "Hay ocm_id repetidos en la tabla de referencia; el cruce con A4 "
        "sería ambiguo"
    )
    referencia = BASE.set_index("ocm_id")["categoria"]

    ETIQUETA_ABSORBIDO = "Absorbido por deduplicación interna en B1"
    A4["categoria_final"] = A4["ocm_id"].map(referencia).fillna(ETIQUETA_ABSORBIDO)

    # Ningún registro de A4 puede quedarse sin destino: o cruza contra una
    # categoría, o se explica por la deduplicación. Cualquier otra ausencia
    # sería un registro perdido entre los dos métodos.
    sin_destino = A4["categoria_final"].isna().sum()
    assert sin_destino == 0, f"{sin_destino} registros de A4 quedaron sin categoría"

    DESCOMPOSICION = pd.crosstab(A4["semaforo"], A4["categoria_final"])
    DESCOMPOSICION = DESCOMPOSICION.reindex(
        ["VERDE", "AMARILLO", "ROJO"]).fillna(0).astype(int)

    print("=" * 78)
    print("DESCOMPOSICIÓN DEL SEMÁFORO DE A4 BAJO LA CLASIFICACIÓN VERIFICADA")
    print("=" * 78)
    print(f"  Registros de A4: {len(A4)}  "
          f"(contraste unidireccional, una fila por estación de OCM)\n")
    print(DESCOMPOSICION.to_string())

    # -------------------------------------------------------------------------
    # La lectura del rojo, que es lo que motivó toda la corrección
    # -------------------------------------------------------------------------
    ROJO = A4[A4["semaforo"] == "ROJO"]
    cobertura = int(ROJO["categoria_final"].str.startswith("C4").sum())
    posicional = int(ROJO["categoria_final"].str.startswith("C3").sum())
    absorbido = int((ROJO["categoria_final"] == ETIQUETA_ABSORBIDO).sum())

    assert cobertura + posicional + absorbido == len(ROJO), (
        "La descomposición del rojo no suma el total de registros rojos"
    )

    print("\n" + "-" * 78)
    print(f"LOS {len(ROJO)} REGISTROS ROJOS DE A4")
    print("-" * 78)
    print(f"  Vacío de cobertura (la otra fuente no registra la estación): {cobertura}")
    print(f"  Desacuerdo posicional real (ambas la registran, lejos):      {posicional}")
    print(f"  Absorbido por deduplicación interna:                         {absorbido}")

    print("\n  Los casos que no son vacío de cobertura:")
    detalle = ROJO[~ROJO["categoria_final"].str.startswith("C4")][
        ["ciudad", "ocm_id", "nombre_ocm", "distancia_m", "categoria_final"]]
    print(detalle.to_string(index=False))

    # Estado de verificación de los casos posicionales: importa distinguir un
    # desacuerdo confirmado de uno que sigue sin conclusión.
    posicionales_ids = ROJO.loc[
        ROJO["categoria_final"].str.startswith("C3"), "ocm_id"]
    estado = ACT[ACT["ocm_id"].isin(posicionales_ids)][
        ["ocm_id", "nombre_ocm", "verificacion_externa", "motivo_verificacion"]]
    print("\n  Estado de verificación de los desacuerdos posicionales:")
    print(estado.to_string(index=False))

DESCOMPOSICIÓN DEL SEMÁFORO DE A4 BAJO LA CLASIFICACIÓN VERIFICADA
  Registros de A4: 38  (contraste unidireccional, una fila por estación de OCM)

categoria_final  Absorbido por deduplicación interna en B1  C1 Coincidencia confirmada  C2 Coincidencia probable  C3 Desacuerdo posicional  C4 Sin contraparte (solo OCM)
semaforo                                                                                                                                                                 
VERDE                                                    0                           3                         0                         0                              0
AMARILLO                                                 0                           2                         9                         0                              0
ROJO                                                     1                           0                         0                         1                             22

-

# Pruebas de aceptación

In [7]:
PRUEBAS = []


def prueba(nombre, condicion, detalle=""):
    PRUEBAS.append((nombre, bool(condicion), detalle))


# --- Integridad de las entradas -------------------------------------------
# Los insumos no pueden haber cambiado mientras corría el cuaderno.
prueba("B1 no fue modificado en disco",
       constantes.sha256_archivo(ARCH_CLASIFICACION)
       == MANIFIESTO_B2["B1_clasificacion_cuatro_categorias.csv"])
prueba("B2_veredictos.json no fue modificado",
       constantes.sha256_archivo(ARCH_VEREDICTOS) == HASH_VEREDICTOS)

# --- Conservación del inventario ------------------------------------------
# Ningún registro puede perderse ni aparecer de la nada: la verificación
# reclasifica, no agrega ni quita estaciones.
prueba("ningún registro OCM se pierde ni aparece",
       set(PREVIA["ocm_id"].dropna()) == set(ACT["ocm_id"].dropna()),
       f"{PREVIA['ocm_id'].nunique()} registros")
prueba("ningún registro OSM se pierde ni aparece",
       set(PREVIA["osm_uid"].dropna()) == set(ACT["osm_uid"].dropna()),
       f"{PREVIA['osm_uid'].nunique()} registros")
prueba("ningún osm_uid aparece dos veces",
       ACT["osm_uid"].dropna().duplicated().sum() == 0)

# --- Alcance de los cambios ------------------------------------------------
# Solo los seis casos declarados pueden moverse. Se comparan todas las
# columnas originales, no solo la categoría: un error de índice al escribir
# podría dejar la categoría intacta y corromper otra columna.
COLUMNAS_B1 = list(PREVIA.columns)
sin_caso_antes = (PREVIA[PREVIA["caso"].isna()][COLUMNAS_B1]
                  .sort_values(COLUMNAS_B1, na_position="last")
                  .reset_index(drop=True))
sin_caso_ahora = (ACT[ACT["caso"].isna()][COLUMNAS_B1]
                  .sort_values(COLUMNAS_B1, na_position="last")
                  .reset_index(drop=True))
prueba("las filas sin caso no cambian en ninguna columna",
       sin_caso_antes.compare(sin_caso_ahora).empty,
       f"{len(sin_caso_antes)} filas")

# --- Tratamiento de los veredictos sin conclusión --------------------------
# El punto más delicado del bloque: None no es False.
no_concluyentes = ACT[ACT["verificacion_externa"] == ESTADO_NO_CONCLUYENTE]
prueba("un veredicto sin conclusión no cambia la categoría",
       all(r["categoria"] == categoria_base(r["categoria_previa"])
           for _, r in no_concluyentes.iterrows()),
       f"{len(no_concluyentes)} casos")
prueba("un veredicto sin conclusión no libera registros",
       (~no_concluyentes["liberado_de_par"]).all())
prueba("None y False no producen el mismo resultado",
       reasignar("C3 Desacuerdo posicional (sin verificar)", None)
       != reasignar("C3 Desacuerdo posicional (sin verificar)", False))

# --- Coherencia de la tabla resultante -------------------------------------
prueba("todo caso lleva su motivo documentado",
       ACT[ACT["caso"].notna()]["motivo_verificacion"].notna().all())
prueba("ninguna categoría conserva el sufijo en el nombre",
       (~ACT["categoria"].str.endswith(SUFIJO_SIN_VERIFICAR)).all())
prueba("la columna de verificación solo toma valores declarados",
       set(ACT["verificacion_externa"].dropna())
       <= {ESTADO_CONFIRMADO, ESTADO_DESCARTADO, ESTADO_NO_CONCLUYENTE})
prueba("solo los registros con caso llevan estado de verificación",
       (ACT["caso"].notna() == ACT["verificacion_externa"].notna()).all())

# --- Pares disueltos --------------------------------------------------------
# Un par descartado tiene que quedar sin rastro del par en la fila OCM y
# haber generado exactamente una fila OSM liberada.
disueltos_ok, disueltos = True, []
for _, r in ACT[(ACT["verificacion_externa"] == ESTADO_DESCARTADO)
                & (~ACT["liberado_de_par"])].iterrows():
    hay_liberado = int((ACT["liberado_de_par"] & (ACT["caso"] == r["caso"])).sum()) == 1
    disueltos_ok &= (pd.isna(r["osm_uid"]) and pd.isna(r["distancia_m"])
                     and hay_liberado)
    disueltos.append(r["caso"])
prueba("cada par disuelto libera exactamente un registro OSM",
       disueltos_ok, ", ".join(disueltos) if disueltos else "ninguno")

# --- Conteos ----------------------------------------------------------------
prueba("el total de registros cuadra con los liberados",
       len(ACT) == len(PREVIA) + len(liberados),
       f"{len(PREVIA)} + {len(liberados)} = {len(ACT)}")
prueba("el conteo de estaciones físicas cuadra",
       fisicas_despues == fisicas_antes + len(liberados),
       f"{fisicas_antes} + {len(liberados)} = {fisicas_despues}")

# --- Rederivación independiente --------------------------------------------
# Segundo camino hacia el mismo resultado: tabla de consulta sobre el JSON
# crudo, sin pasar por 'reasignar'.
TABLA_CONTROL = {
    ("C3", True): "C3 Desacuerdo posicional",
    ("C3", False): "C4 Sin contraparte (solo OCM)",
    ("C5", True): "C5 Punto secundario del mismo sitio",
    ("C5", False): "C4 Sin contraparte (solo OSM)",
}
with open(ARCH_VEREDICTOS, encoding="utf-8") as fh:
    VEREDICTOS_CRUDOS = json.load(fh)["veredictos"]

rederivacion_ok = True
for _, r in ACT[ACT["caso"].notna() & (~ACT["liberado_de_par"])].iterrows():
    v = VEREDICTOS_CRUDOS[r["caso"]]
    familia = categoria_base(r["categoria_previa"])[:2]
    esperada = (categoria_base(r["categoria_previa"]) if v is None
                else TABLA_CONTROL[(familia, bool(v))])
    rederivacion_ok &= (esperada == r["categoria"])
prueba("la categoría final se rederiva desde el JSON crudo", rederivacion_ok,
       f"{int(ACT['caso'].notna().sum()) - len(liberados)} casos")

# --- Resultado --------------------------------------------------------------
print("=" * 78)
print("PRUEBAS DE ACEPTACIÓN")
print("=" * 78)
for nombre, ok, detalle in PRUEBAS:
    marca = "OK   " if ok else "FALLA"
    print(f"  [{marca}] {nombre}" + (f"   ({detalle})" if detalle else ""))

fallidas = [n for n, ok, _ in PRUEBAS if not ok]
print("-" * 78)
print(f"  {len(PRUEBAS) - len(fallidas)} de {len(PRUEBAS)} pasan")

if fallidas:
    raise AssertionError("Pruebas fallidas: " + "; ".join(fallidas))

PRUEBAS DE ACEPTACIÓN
  [OK   ] B1 no fue modificado en disco
  [OK   ] B2_veredictos.json no fue modificado
  [OK   ] ningún registro OCM se pierde ni aparece   (37 registros)
  [OK   ] ningún registro OSM se pierde ni aparece   (52 registros)
  [OK   ] ningún osm_uid aparece dos veces
  [OK   ] las filas sin caso no cambian en ninguna columna   (66 filas)
  [OK   ] un veredicto sin conclusión no cambia la categoría   (4 casos)
  [OK   ] un veredicto sin conclusión no libera registros
  [OK   ] None y False no producen el mismo resultado
  [OK   ] todo caso lleva su motivo documentado
  [OK   ] ninguna categoría conserva el sufijo en el nombre
  [OK   ] la columna de verificación solo toma valores declarados
  [OK   ] solo los registros con caso llevan estado de verificación
  [OK   ] cada par disuelto libera exactamente un registro OSM   (C3-2, C3-3)
  [OK   ] el total de registros cuadra con los liberados   (72 + 2 = 74)
  [OK   ] el conteo de estaciones físicas cuadra   (69 + 2 = 7

# Guardado

In [8]:
COLUMNAS_SALIDA = [
    "ciudad", "categoria", "origen",
    "ocm_id", "osm_uid", "nombre_ocm", "nombre_osm",
    "distancia_m", "similitud_nombre", "similitud_operador",
    "caso", "categoria_previa", "veredicto",
    "verificacion_externa", "motivo_verificacion", "liberado_de_par",
]
assert set(COLUMNAS_SALIDA) == set(ACT.columns), (
    "El orden de columnas declarado no coincide con la tabla: "
    f"faltan {set(ACT.columns) - set(COLUMNAS_SALIDA)}, "
    f"sobran {set(COLUMNAS_SALIDA) - set(ACT.columns)}"
)

SALIDA = ACT[COLUMNAS_SALIDA]

archivos = {}


def escribir_csv(tabla, nombre, index=False):
    ruta = RUTA_OUT / nombre
    tabla.to_csv(ruta, index=index, encoding="utf-8-sig")
    archivos[nombre] = ruta
    return ruta


escribir_csv(SALIDA, "B2_1_clasificacion_verificada.csv")
escribir_csv(CONTRASTE, "B2_1_contraste_B1_B2_1.csv", index=True)
escribir_csv(MOVIMIENTOS, "B2_1_movimientos_por_caso.csv")
escribir_csv(CONTEOS, "B2_1_conteos_absolutos.csv", index=True)
if DESCOMPOSICION is not None:
    escribir_csv(DESCOMPOSICION, "B2_1_descomposicion_semaforo_A4.csv", index=True)

# -----------------------------------------------------------------------------
# Manifiesto de insumos
#
# Mismo formato que el de B2, para que la cadena se pueda recorrer hacia
# atrás archivo por archivo hasta los crudos congelados de A2 y A3. Las
# entradas 'crudo::' se heredan sin recalcular: B2 ya las validó.
# -----------------------------------------------------------------------------
MANIFIESTO = {
    "B1_clasificacion_cuatro_categorias.csv": constantes.sha256_archivo(ARCH_CLASIFICACION),
    "B1_pares_primarios.csv": constantes.sha256_archivo(ARCH_PARES),
    "B2_veredictos.json": HASH_VEREDICTOS,
    "B2_manifiesto_insumos.json": constantes.sha256_archivo(ARCH_MANIFIESTO_B2),
}
if Path(ARCH_A4).exists():
    MANIFIESTO["A4_resultado_final_OCM_OSM.csv"] = constantes.sha256_archivo(ARCH_A4)
MANIFIESTO.update(CRUDOS_HEREDADOS)

ruta_manifiesto = RUTA_OUT / "B2_1_manifiesto_insumos.json"
with open(ruta_manifiesto, "w", encoding="utf-8") as fh:
    json.dump(MANIFIESTO, fh, indent=2, ensure_ascii=False, sort_keys=True)

# -----------------------------------------------------------------------------
# Auditoría del procesamiento
#
# Deja escrito, junto a los datos, qué método produjo este resultado y bajo
# qué decisiones. Quien abra la carpeta dentro de un año no tiene por qué
# leer el cuaderno para saberlo.
# -----------------------------------------------------------------------------
AUDITORIA = {
    "bloque": "B2.1",
    "proposito": (
        "aplicar a la clasificación de B1 los veredictos de verificación "
        "externa emitidos por B2, y documentar el contraste entre ambas"
    ),
    "verificacion_externa": {
        "fuente": FICHA["fuente"]["fuente"],
        "url": FICHA["fuente"]["url"],
        "version": FICHA["fuente"]["version"],
        "licencia": FICHA["fuente"]["licencia"],
        "sha256_fuente": FICHA["fuente"]["sha256"],
        "fecha_verificacion": FICHA["fecha_verificacion"],
        "metodo": FICHA["metodo"],
        "tolerancia_adoptada_m": FICHA["tolerancia_adoptada_m"],
        "banda_tolerancia_m": FICHA["banda_tolerancia_m"],
        "area_max_manzana_m2": FICHA["area_max_manzana_m2"],
    },
    "decisiones": {
        "estado_de_verificacion": (
            "se reporta en la columna 'verificacion_externa' y no dentro del "
            "nombre de la categoría, para que una misma categoría no se "
            "parta en dos al contar"
        ),
        "veredicto_sin_conclusion": (
            "None no equivale a False: que la manzana no discrimine es "
            "ausencia de evidencia, no evidencia de que sean estaciones "
            "distintas. Estos casos conservan su categoría de B1"
        ),
        "registro_liberado": (
            "al disolverse un par C3 el registro OSM pasa a 'solo OSM' y NO "
            "se vuelve a emparejar: el emparejamiento mutuo es competencia "
            "de B1 y reabrirlo desde este cuaderno rompería la linealidad "
            "del pipeline"
        ),
        "A4": (
            "se cruza por ocm_id para descomponer su semáforo; A4 no se "
            "recalcula y permanece congelado como registro del método "
            "unidireccional"
        ),
    },
    "veredictos_aplicados": {
        caso: {
            "veredicto": VEREDICTOS[caso],
            "estado": (ESTADO_NO_CONCLUYENTE if VEREDICTOS[caso] is None
                       else (ESTADO_CONFIRMADO if VEREDICTOS[caso]
                             else ESTADO_DESCARTADO)),
            "motivo": FICHA["motivos"][caso],
        }
        for caso in sorted(VEREDICTOS)
    },
    "conteos": {
        "registros_B1": len(PREVIA),
        "registros_B2_1": len(ACT),
        "registros_liberados": len(liberados),
        "estaciones_fisicas_B1": int(fisicas_antes),
        "estaciones_fisicas_B2_1": int(fisicas_despues),
        "estaciones_fisicas_por_ciudad": {
            ciudad: int(n) for ciudad, n in por_ciudad.items()
        },
        "categorias": {k: int(v) for k, v in
                       ACT["categoria"].value_counts().sort_index().items()},
        "casos_declarados": int(len(MOVIMIENTOS)),
        "casos_con_conclusion": int(resueltos),
        "casos_que_cambian_de_categoria": int(movidos),
    },
    "pruebas_de_aceptacion": {
        "total": len(PRUEBAS),
        "pasan": len(PRUEBAS) - len(fallidas),
        "nombres": [n for n, _, _ in PRUEBAS],
    },
    "limitaciones": [
        "la verificación se apoya en la capa catastral de manzanas de Bogotá; "
        "los casos de Medellín no tienen una capa equivalente evaluada",
        "cuatro de los seis casos declarados siguen sin conclusión y su "
        "categoría es la de B1, no una categoría verificada",
    ],
}

ruta_auditoria = RUTA_OUT / "B2_1_auditoria_procesamiento.json"
with open(ruta_auditoria, "w", encoding="utf-8") as fh:
    json.dump(AUDITORIA, fh, indent=2, ensure_ascii=False)

# -----------------------------------------------------------------------------
# Registro de lo escrito
# -----------------------------------------------------------------------------
print("=" * 78)
print("SALIDAS DE B2.1")
print("=" * 78)
print(f"  Carpeta: {RUTA_OUT}\n")

for nombre, ruta in archivos.items():
    print(f"  {nombre}")
    print(f"      {constantes.sha256_archivo(ruta)[:16]}…   "
          f"{ruta.stat().st_size:,} bytes")

for ruta in (ruta_manifiesto, ruta_auditoria):
    print(f"  {ruta.name}")
    print(f"      {constantes.sha256_archivo(ruta)[:16]}…   "
          f"{ruta.stat().st_size:,} bytes")

print("\n  B1, B2 y A4 quedan intactos: este cuaderno solo escribió en B2.1.")
print(f"  Archivo canónico para los bloques siguientes: "
      f"B2_1_clasificacion_verificada.csv ({len(SALIDA)} registros, "
      f"{fisicas_despues} estaciones físicas).")

# -----------------------------------------------------------------------------
# Verificación de ida y vuelta
#
# Las pruebas de aceptación validan la tabla en memoria. Esto valida el
# archivo, que es cosa distinta: el error del '475409.0' que rompió una
# verificación de la tutoría no estaba en los datos, estaba en la escritura.
# Como este CSV es lo que van a consumir B3 y B4, se relee y se compara.
# -----------------------------------------------------------------------------
RELEIDO = pd.read_csv(
    archivos["B2_1_clasificacion_verificada.csv"],
    encoding="utf-8-sig",
    dtype={"ocm_id": "string", "osm_uid": "string", "caso": "string",
           "verificacion_externa": "string"},
)

assert len(RELEIDO) == len(SALIDA), (
    f"El archivo escrito tiene {len(RELEIDO)} filas y la tabla {len(SALIDA)}"
)
assert list(RELEIDO.columns) == COLUMNAS_SALIDA, (
    "Las columnas cambiaron al escribir"
)
assert set(RELEIDO["ocm_id"].dropna()) == set(SALIDA["ocm_id"].dropna()), (
    "Los identificadores OCM no sobrevivieron la escritura"
)
assert set(RELEIDO["osm_uid"].dropna()) == set(SALIDA["osm_uid"].dropna()), (
    "Los identificadores OSM no sobrevivieron la escritura"
)
assert not RELEIDO["ocm_id"].dropna().str.endswith(".0").any(), (
    "Un identificador se escribió como decimal"
)
conteos_releidos = RELEIDO["categoria"].value_counts().to_dict()
conteos_memoria = SALIDA["categoria"].value_counts().to_dict()
assert conteos_releidos == conteos_memoria, (
    f"Los conteos por categoría no coinciden tras releer el archivo: "
    f"{conteos_releidos} vs {conteos_memoria}"
)

print(f"\n  Relectura del archivo canónico: {len(RELEIDO)} filas, "
      f"identificadores y conteos intactos.")

SALIDAS DE B2.1
  Carpeta: F:\Tesis\Bloque B\Entregable B2.1

  B2_1_clasificacion_verificada.csv
      b0a2e926cf09b73c…   11,938 bytes
  B2_1_contraste_B1_B2_1.csv
      f68aca77b5b9480c…   278 bytes
  B2_1_movimientos_por_caso.csv
      271ff2eb46a32a99…   1,786 bytes
  B2_1_conteos_absolutos.csv
      723f1138a1cbb625…   255 bytes
  B2_1_descomposicion_semaforo_A4.csv
      26234b724e8676dd…   217 bytes
  B2_1_manifiesto_insumos.json
      a988d49e8b2bffab…   880 bytes
  B2_1_auditoria_procesamiento.json
      5d8abae172509073…   5,269 bytes

  B1, B2 y A4 quedan intactos: este cuaderno solo escribió en B2.1.
  Archivo canónico para los bloques siguientes: B2_1_clasificacion_verificada.csv (74 registros, 71 estaciones físicas).

  Relectura del archivo canónico: 74 filas, identificadores y conteos intactos.
